### **1. Importaciones y Carga (Preprocesamiento)**

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('rent_features.csv')
print(f"Registros totales: {len(df)}")

df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
df = df.dropna(subset=['latitud', 'longitud'])
df = df[(df['latitud'] != 0) & (df['longitud'] != 0)]

columnas_modelo = [
    'm2_terreno', 'm2_construido', 'banos', 'medios_banos',
    'estacionamientos', 'antiguedad',
    'amen_Alberca', 'amen_Cocina integral', 'amen_Amueblado', 'amen_Elevador',
    'amen_Cuartos de servicio', 'tipo_propiedad_departamento', 'latitud', 'longitud'
]

X = df[columnas_modelo].copy()
y = df['precio']

y_log = np.log1p(y)

strat_col = df['tipo_propiedad_original'].copy()

indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=strat_col
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_log, y_test_log = y_log.iloc[train_idx], y_log.iloc[test_idx] 

print(f"Entrenamiento: {X_train.shape[0]} registros, Prueba: {X_test.shape[0]} registros")

Registros totales: 3097
Entrenamiento: 2477 registros, Prueba: 620 registros


### **2. Multicolinealidad (VIF)**

In [18]:
X_vif = X_train.copy()
X_vif_const = add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif_const.values, i) 
                    for i in range(X_vif_const.shape[1])]

print("\n--- Factor de Inflación de Varianza (VIF) ---")
print(vif_data.sort_values('VIF', ascending=False))


--- Factor de Inflación de Varianza (VIF) ---
                       Variable         VIF
0                         const  205.051427
2                 m2_construido    3.502217
3                         banos    2.282544
12  tipo_propiedad_departamento    1.889460
5              estacionamientos    1.618641
4                  medios_banos    1.496503
1                    m2_terreno    1.488075
10                amen_Elevador    1.409795
11     amen_Cuartos de servicio    1.389428
13                      latitud    1.332557
14                     longitud    1.320133
7                  amen_Alberca    1.172913
6                    antiguedad    1.157073
9                amen_Amueblado    1.117979
8          amen_Cocina integral    1.103996


### **3. Capacidad de los Modelos de árbol.**

In [22]:
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_base = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)

scores_rf = -cross_val_score(rf_base, X_train, y_train_log, cv=5, scoring='neg_root_mean_squared_error')
scores_xgb = -cross_val_score(xgb_base, X_train, y_train_log, cv=5, scoring='neg_root_mean_squared_error')

def rmse_pesos(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(np.expm1(y_true_log), np.expm1(y_pred_log)))

scores_rf_pesos = -cross_val_score(rf_base, X_train, y_train_log, cv=5,
                                    scoring=make_scorer(rmse_pesos, greater_is_better=False))
scores_xgb_pesos = -cross_val_score(xgb_base, X_train, y_train_log, cv=5,
                                    scoring=make_scorer(rmse_pesos, greater_is_better=False))


print(f"RMSE CV (log) - Random Forest Base: {scores_rf.mean():.4f} +/- {scores_rf.std():.4f}")
print(f"RMSE CV (MXN) - Random Forest Base: ${scores_rf_pesos.mean():,.2f} ± ${scores_rf_pesos.std():,.2f}")
print(f"\nRMSE CV (log) - XGBoost Base: {scores_xgb.mean():.4f} +/- {scores_xgb.std():.4f}")
print(f"RMSE CV (MXN) - XGBoost Base: ${scores_xgb_pesos.mean():,.2f} ± ${scores_xgb_pesos.std():,.2f}")

RMSE CV (log) - Random Forest Base: 0.3126 +/- 0.0131
RMSE CV (MXN) - Random Forest Base: $19,471.54 ± $1,375.11

RMSE CV (log) - XGBoost Base: 0.2932 +/- 0.0128
RMSE CV (MXN) - XGBoost Base: $18,941.61 ± $1,514.82


### **4. Optimización de Hiperparámetros (GridSearch)**

#### **4.1 Random Forest (Regularizado)**

In [21]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid_rf = {
    'n_estimators': [150, 250],
    'max_depth': [8, 10, 12],       # Limitado para evitar sobreajuste a ubicación
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
    'max_features': [0.5, 0.7, 'sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5,
                        scoring='neg_root_mean_squared_error',
                        verbose=1, n_jobs=-1)
grid_rf.fit(X_train, y_train_log)

print("\n--- Mejores parámetros para Random Forest ---")
print(grid_rf.best_params_)
print(f"Mejor RMSE CV (log): {-grid_rf.best_score_:.4f}")

cv_rf_opt_pesos = -cross_val_score(grid_rf.best_estimator_, X_train, y_train_log, cv=5,
                                    scoring=make_scorer(rmse_pesos, greater_is_better=False))
print(f"\nMejor RMSE CV (MXN): ${cv_rf_opt_pesos.mean():,.2f} ± ${cv_rf_opt_pesos.std():,.2f}")

Fitting 5 folds for each of 72 candidates, totalling 360 fits

--- Mejores parámetros para Random Forest ---
{'max_depth': 12, 'max_features': 0.7, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 250}
Mejor RMSE CV (log): 0.3130

Mejor RMSE CV (MXN): $19,938.50 ± $1,560.80


#### **4.2 XGBoost (con Regularización L1/L2 explícita)**

In [23]:
xgb = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)

param_grid_xgb = {
    'n_estimators': [300, 500],
    'learning_rate': [0.03, 0.05],
    'max_depth': [4, 6],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'reg_alpha': [0, 0.1, 1],       # Regularización L1 (apaga variables ruidosas)
    'reg_lambda': [1, 5, 10]        # Regularización L2 (reduce impacto de extremos)
}

grid_xgb = GridSearchCV(xgb, param_grid_xgb, cv=5,
                        scoring='neg_root_mean_squared_error',
                        verbose=1, n_jobs=-1)
grid_xgb.fit(X_train, y_train_log)

print("\n--- Mejores parámetros para XGBoost ---")
print(grid_xgb.best_params_)
print(f"Mejor RMSE CV (log): {-grid_xgb.best_score_:.4f}")

cv_xgb_opt_pesos = -cross_val_score(grid_xgb.best_estimator_, X_train, y_train_log, cv=5,
                                    scoring=make_scorer(rmse_pesos, greater_is_better=False))
print(f"\nMejor RMSE CV (MXN): ${cv_xgb_opt_pesos.mean():,.2f} ± ${cv_xgb_opt_pesos.std():,.2f}")

Fitting 5 folds for each of 288 candidates, totalling 1440 fits

--- Mejores parámetros para XGBoost ---
{'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'reg_alpha': 0.1, 'reg_lambda': 10, 'subsample': 0.8}
Mejor RMSE CV (log): 0.2781

Mejor RMSE CV (MXN): $18,195.37 ± $1,626.15


### **5. Elección del mejor modelo.**

In [24]:
best_rf = grid_rf.best_estimator_
best_xgb = grid_xgb.best_estimator_

importancias_rf = pd.DataFrame({
    'Variable': X_train.columns,
    'Importancia': best_rf.feature_importances_
}).sort_values('Importancia', ascending=False)

importancias_xgb = pd.DataFrame({
    'Variable': X_train.columns,
    'Importancia': best_xgb.feature_importances_
}).sort_values('Importancia', ascending=False)

print("\n--- Top 5 Importancias (Random Forest) ---")
print(importancias_rf.head(5))

print("\n--- Top 5 Importancias (XGBoost) ---")
print(importancias_xgb.head(5))


--- Top 5 Importancias (Random Forest) ---
         Variable  Importancia
1   m2_construido     0.491990
2           banos     0.095351
13       longitud     0.084262
12        latitud     0.082191
0      m2_terreno     0.070808

--- Top 5 Importancias (XGBoost) ---
                       Variable  Importancia
11  tipo_propiedad_departamento     0.278920
1                 m2_construido     0.192240
2                         banos     0.114792
10     amen_Cuartos de servicio     0.078366
9                 amen_Elevador     0.074645


### **6. Conclusión**

In [25]:
print("\n" + "="*50)
print("CONCLUSIÓN")
print("="*50)
print(f"1. Regresión Lineal descartada: Aunque la colinealidad no afecta significativamente la Heterocedasticidad"
        " demuestra que el problema no es lineal dado el abanico de puntos.")
print(f"2. Random Forest mejora con regularización (parámetros encontrados: {grid_rf.best_params_}).")
print(f"3. XGBoost muestra el mejor equilibrio (parámetros: {grid_xgb.best_params_}).")
print(f"4. Se procederá a entrenar el modelo final XGBoost con estos parámetros sobre TODO X_train,")
print(f"   para evaluar en X_test y obtener las métricas definitivas (MAE, RMSE, R², MAPE).")


CONCLUSIÓN
1. Regresión Lineal descartada: Aunque la colinealidad no afecta significativamente la Heterocedasticidad demuestra que el problema no es lineal dado el abanico de puntos.
2. Random Forest mejora con regularización (parámetros encontrados: {'max_depth': 12, 'max_features': 0.7, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 250}).
3. XGBoost muestra el mejor equilibrio (parámetros: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'reg_alpha': 0.1, 'reg_lambda': 10, 'subsample': 0.8}).
4. Se procederá a entrenar el modelo final XGBoost con estos parámetros sobre TODO X_train,
   para evaluar en X_test y obtener las métricas definitivas (MAE, RMSE, R², MAPE).
